In [3]:
import os
import json
import numpy as np
import pandas as pd


# --- Configuration Options ---
RANK_CORRELATION_METRIC = "pearson"   # Options: "pearson", "spearman", "kendall"
TUNING_REGIME = "lambda_tuned"        # Options: "lambda_tuned", "epoch_tuned"
BOLD_BEST = True                      # Bold the highest score in each shot column
BOLD_OURS = False                     # Set True to bold the entire row for proposed metrics
DECIMALS = 4

SHOTS_ORDER = [1, 5, 10, 15, 20, 25]
ROW_ORDER = [
    "LogME",
    "LFC",
    "LGC",
    "LGC - CLS",
    "SVM-margin",
    "SVM-LOO-Margin",
]
OURS_ROWS = {"LGC - CLS", "SVM-margin", "SVM-LOO-Margin"}

ALIASES = {
    "LogME": ["LogME", "logme"],
    "LFC": ["LFC", "lfc"],
    "LGC": ["LGC", "lgc"],
    "LGC - CLS": ["LGC-CLS", "LGC - CLS", "LGC-CLS_new", "LGC_cls"],
    "SVM-margin": ["SVM-Margin", "SVM-margin", "SVM-Margin_new", "SVM_margin"],
    "SVM-LOO-Margin": ["SVM-LOO-Margin", "SVM-LOO-margin", "SVM-LOO-Margin_new", "SVM_loo_margin"],
}

METRIC_CONFIG = {
    "pearson": ("pearson_r", "Pearson Correlation ($r$)", "r"),
    "spearman": ("spearman_rho", "Spearman Rank Correlation ($\\rho$)", "rho"),
    "kendall": ("kendall_tau", "Kendall Rank Correlation ($\\tau$)", "tau"),
}


def find_correlation_file(tuning: str) -> str:
    for root in ["./rank_correlation_results", "../rank_correlation_results"]:
        for ext in [".json", ".csv"]:
            candidate = os.path.join(root, f"rank_correlations_{tuning}{ext}")
            if os.path.exists(candidate):
                return candidate
    raise FileNotFoundError(
        f"Could not find rank_correlations_{tuning}.json/.csv in ./rank_correlation_results or ../rank_correlation_results."
    )


def load_correlation_table(file_path: str, value_key: str) -> pd.DataFrame:
    ext = os.path.splitext(file_path)[1].lower()
    if ext == ".json":
        with open(file_path, "r") as f:
            raw = json.load(f)
        records_df = pd.DataFrame(raw["correlations"])
    elif ext == ".csv":
        records_df = pd.read_csv(file_path)
    else:
        raise ValueError(f"Unsupported file format: {ext}")

    records_df["shots"] = records_df["shots"].astype(int)
    records_df[value_key] = pd.to_numeric(records_df[value_key], errors="coerce")
    return records_df.pivot(index="metric", columns="shots", values=value_key)


def generate_latex_table(
    rank_correlation_metric: str = "pearson",
    tuning_regime: str = "lambda_tuned",
    file_path: str = None,
    bold_best: bool = True,
    bold_ours: bool = False,
    decimals: int = 4,
) -> str:
    metric_key = rank_correlation_metric.lower().strip()
    if metric_key not in METRIC_CONFIG:
        raise ValueError(f"rank_correlation_metric must be one of {list(METRIC_CONFIG.keys())}")
    if tuning_regime not in ("lambda_tuned", "epoch_tuned"):
        raise ValueError("tuning_regime must be 'lambda_tuned' or 'epoch_tuned'")

    if file_path is None:
        file_path = find_correlation_file(tuning_regime)

    value_col, metric_title, metric_short = METRIC_CONFIG[metric_key]
    pivoted_df = load_correlation_table(file_path, value_col)

    ordered_rows = {}
    for display_name in ROW_ORDER:
        matched_name = next((a for a in ALIASES[display_name] if a in pivoted_df.index), None)
        if matched_name is None:
            raise KeyError(
                f"Metric '{display_name}' not found in {file_path}. Available: {list(pivoted_df.index)}"
            )
        ordered_rows[display_name] = [
            float(pivoted_df.loc[matched_name, s]) if s in pivoted_df.columns else np.nan
            for s in SHOTS_ORDER
        ]

    table_df = pd.DataFrame.from_dict(ordered_rows, orient="index", columns=SHOTS_ORDER)
    col_max = table_df.max(axis=0, skipna=True)

    tuning_latex = tuning_regime.replace("_", "\\_")
    col_spec = "|" + "|".join(["c"] * (len(SHOTS_ORDER) + 1)) + "|"
    shot_headers = " & ".join([f"{s}-shot" for s in SHOTS_ORDER])

    lines = [
        "% Requires \\usepackage{graphicx}, \\usepackage{caption}, \\usepackage[dvipsnames]{xcolor}",
        "\\begin{center}",
        "\t\\scalebox{1.1}{",
        f"\t\t\\begin{{tabular}}{{{col_spec}}}",
        "\t\t\t\\hline",
        f"\t\t\tMethods & {shot_headers} \\\\",
        "\t\t\t\\hline",
    ]

    for row_name in ROW_ORDER:
        is_ours = bold_ours and (row_name in OURS_ROWS)
        row_label = f"\\textbf{{{row_name}}}" if is_ours else row_name

        cells = []
        for s in SHOTS_ORDER:
            val = table_df.loc[row_name, s]
            if pd.isna(val):
                cell_str = "--"
            else:
                cell_str = f"{val:.{decimals}f}"
                if is_ours or (bold_best and np.isclose(val, col_max[s], atol=1e-12)):
                    cell_str = f"\\textbf{{{cell_str}}}"
            cells.append(cell_str)

        lines.append(f"\t\t\t{row_label} & {' & '.join(cells)} \\\\")

    lines.extend([
        "\t\t\t\\hline",
        "\t\t\\end{tabular}",
        "\t}\\\\",
        f"\t\\captionof{{table}}{{\\color{{Blue}} {metric_title} across $K$-shot regimes (\\texttt{{{tuning_latex}}})}}",
        f"\t\\label{{tab:{metric_short}_{tuning_regime}}}",
        "\t\\vspace{-0.25in}",
        "\\end{center}",
        "\\vspace{0.25in}",
    ])

    return "\n".join(lines)


# --- Run in Notebook ---
latex_code = generate_latex_table(
    rank_correlation_metric=RANK_CORRELATION_METRIC,
    tuning_regime=TUNING_REGIME,
    file_path=None,
    bold_best=BOLD_BEST,
    bold_ours=BOLD_OURS,
    decimals=DECIMALS,
)
print(latex_code)

% Requires \usepackage{graphicx}, \usepackage{caption}, \usepackage[dvipsnames]{xcolor}
\begin{center}
	\scalebox{1.1}{
		\begin{tabular}{|c|c|c|c|c|c|c|}
			\hline
			Methods & 1-shot & 5-shot & 10-shot & 15-shot & 20-shot & 25-shot \\
			\hline
			LogME & -0.2352 & 0.7354 & 0.7684 & \textbf{0.9586} & 0.9411 & \textbf{0.9432} \\
			LFC & 0.1011 & 0.8171 & 0.7705 & 0.7075 & 0.8006 & 0.8167 \\
			LGC & 0.1011 & 0.8171 & 0.7705 & 0.7075 & 0.8006 & 0.8167 \\
			LGC - CLS & \textbf{0.5440} & \textbf{0.8493} & \textbf{0.8495} & 0.7245 & 0.8450 & 0.8499 \\
			SVM-margin & 0.1227 & 0.7458 & 0.8123 & 0.8348 & 0.9021 & 0.9264 \\
			SVM-LOO-Margin & -- & 0.5122 & 0.7130 & 0.8996 & \textbf{0.9435} & 0.9053 \\
			\hline
		\end{tabular}
	}\\
	\captionof{table}{\color{Blue} Pearson Correlation ($r$) across $K$-shot regimes (\texttt{lambda\_tuned})}
	\label{tab:r_lambda_tuned}
	\vspace{-0.25in}
\end{center}
\vspace{0.25in}
